#### Context Length Setting

Check context length of qwen3-0.6b, and longest len of my chunks

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-0.6B')

In [ ]:
from custom_textsplitter import CustomTextSplitter
import pandas as pd

chat_df = pd.read_csv('data/processed_data/whatsapp_chats.csv')
splitter = CustomTextSplitter()

chunk_df = splitter.split_messages(message_df=chat_df,chunk_size=1000,return_df=True)
chunk_df['tokenized_chunks_len'] = chunk_df['chunk_text'].apply(tokenizer.encode).apply(len)

In [11]:
chunk_df['tokenized_chunks_len'].describe()

count    5085.000000
mean      306.621042
std        59.305509
min         8.000000
25%       269.000000
50%       300.000000
75%       342.000000
max       691.000000
Name: tokenized_chunks_len, dtype: float64

#### Enhance Metadata Filtering 

In [ ]:
import ast
import chromadb


## Add other_person name for pre-filtering chats in Chroma
client = chromadb.PersistentClient('.chroma_db')
collection = client.get_collection('chat_documents')


In [ ]:

all_docs = collection.get(include=["metadatas"])
other_persons = []

for doc in all_docs['metadatas']:

    is_gc = doc['is_groupchat']
    if not is_gc:
        participants = ast.literal_eval(doc['participants'])
        other_person = [person for person in participants if person!='Dan'][0]
    else:
        other_person='N/A'

    doc['other_person'] = other_person
    ## separate thing add source as whatsapp
    doc['source'] = 'whatsapp'

    other_persons.append(doc)


collection.update(
    ids=all_docs['ids'],
    metadatas=other_persons
)

In [ ]:
from datetime import datetime, date

metadatas = []

for doc in all_docs['metadatas']:
    date_range_str = doc['date_range']

    start_str, end_str = date_range_str.split(' - ')
    start_date = datetime.strptime(start_str.strip(), '%Y-%m-%d').date()
    end_date = datetime.strptime(end_str.strip(), '%Y-%m-%d').date()

    start_date=int(start_date.strftime("%Y%m%d"))
    end_date= int(end_date.strftime("%Y%m%d"))

    doc['start_date'] = start_date
    doc['end_date'] = end_date

    metadatas.append(doc)

collection.update(
    all_docs['ids'],
    metadatas=metadatas
)

In [ ]:
filter_date = datetime.strptime("2023-02-01",'%Y-%m-%d')
filter_date = int(filter_date.strftime("%Y%m%d"))

documents = collection.query(
    "NonaSSA gc messages", 
    k=5, 
    filter={
        "$and": [
        {"start_date" : {"$lte" : filter_date}},
        {"other_person": "Damian"}
        ]})
first_document = documents[1] if documents else None
first_document.metadata

### Merge WhatsApp and Instagram Collections

In [1]:
import ast
import chromadb


## Add other_person name for pre-filtering chats in Chroma
client = chromadb.PersistentClient('.chroma_db')
collection_wa = client.get_collection('chat_documents')
collection_ig = client.get_collection('chat_documents_instagram')

In [12]:
import ast
import chromadb


## Add other_person name for pre-filtering chats in Chroma
client = chromadb.PersistentClient('.chroma_db')
collection_ig = client.get_collection('chat_documents_instagram_Qwen3-Embedding-4B-Q4KM-latest')
all_docs = collection_ig.get(include=["metadatas"])

In [18]:
client = chromadb.PersistentClient('.chroma_db')
collection_ig = client.get_collection('chat_documents_instagram_Qwen3-Embedding-0.6B-Q8_0-latest')

In [16]:
collection_ig.modify(name='chat_documents_instagram_Qwen3-Embedding-0.6B-Q8_0-latest')

In [21]:
collection_name = 'chat_documents_instagram'
model_name = 'Qwen3-Embedding-4B-Q4KM:latest'

collection_name = collection_name + '_' + model_name.replace(':','-')
collection_ig = client.get_collection(collection_name)

#### Trial for the Reranker Model